In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os, sys
sys.path.append(os.path.abspath(".."))

In [0]:
# 2. imports and the stream settings

from pipeline_utils.silver.process_redemptions_batch import load_redemption_batch_to_silver



In [0]:
BRONZE_TABLE    = "skypoints.bronze.raw_redemptions"
CHECKPOINT_PATH = "/Volumes/skypoints/silver/redemptions/redemptions_checkpoints"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS skypoints.silver.silver_redemptions
(
    member_id              STRING,
    txn_id                 STRING,
    txn_date               DATE,
    partner                STRING,
    miles_redeemed         INT,
    status                 STRING,
    feed_date              DATE,
    is_member_id_orphan       BOOLEAN   COMMENT 'Member Id is yet to arrived' ,
    file_date              DATE      COMMENT 'Date associated with the source file; tie-breaker.',
    ingestion_timestamp    TIMESTAMP COMMENT 'Bronze ingestion timestamp; tie-breaker.',
    source_file_name       STRING    COMMENT 'Source file name for data lineage.'
)
USING DELTA
COMMENT 'Silver redemptions: validated and standardized redemption transactions.';


In [0]:
%sql

CREATE TABLE IF NOT EXISTS skypoints.silver.redemption_rejects
(
    member_id           STRING,
    reject_reason       STRING,
    _rescued_data       STRING,
    source_file_name    STRING,
    file_date           DATE      COMMENT 'Source file date in YYYY-MM-DD format.',
    ingestion_timestamp TIMESTAMP COMMENT 'Bronze load time of the rejected record.',
    rejected_at         TIMESTAMP COMMENT 'When silver rejected the record.'
)
USING DELTA
COMMENT 'Silver quarantine for redemption records that fail validation. Nothing is dropped silently.';


In [0]:
# 5. process everything new in bronze, then stop
query = (
    spark.readStream.table(BRONZE_TABLE)
    .writeStream
    .foreachBatch(load_redemption_batch_to_silver)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

In [0]:
%sql
select * from skypoints.silver.silver_redemptions 
inner join skypoints.silver.member_country_lookup
on skypoints.silver.silver_redemptions.member_id = skypoints.silver.member_country_lookup.member_id